# Build a Vamana index with cuVS and search it with DiskANN

This notebook demonstrates an end-to-end workflow that combines GPU-accelerated index construction in [cuVS](https://github.com/rapidsai/cuvs) with CPU search via [DiskANN](https://github.com/microsoft/DiskANN):

1. Generate a random float32 dataset.
2. Build a Vamana graph index on the GPU using `cuvs.neighbors.vamana`.
3. Serialize the index to disk in the **in-memory DiskANN** file format.
4. Load the serialized index with `diskannpy.StaticMemoryIndex` and run nearest-neighbor queries.

## Requirements

- A CUDA-capable GPU and a working `cuvs` Python install.
- The `diskannpy` Python package (`pip install diskannpy`). DiskANN currently ships wheels for Python 3.9 - 3.11 only.
- `numpy`.

The dataset is passed to `vamana.build` as a NumPy (host) array; cuVS will copy it to the GPU internally, so `cupy` is not required for this notebook.

> **Note:** cuVS Vamana currently only supports *building* and *serializing* the graph. Search is delegated to DiskANN's CPU implementation.

In [4]:
import os
import shutil
import tempfile

import numpy as np

from cuvs.neighbors import vamana

import diskannpy

## 1. Create a synthetic dataset

We use a small random float32 dataset so the notebook runs quickly. Vamana also supports `int8` and `uint8` inputs.

In [5]:
rng = np.random.default_rng(seed=42)

n_samples = 50_000
n_features = 64
n_queries = 100
k = 10

dataset = rng.random((n_samples, n_features), dtype=np.float32)
queries = rng.random((n_queries, n_features), dtype=np.float32)

print(f"dataset shape: {dataset.shape}, dtype: {dataset.dtype}")
print(f"queries shape: {queries.shape}")

dataset shape: (50000, 64), dtype: float32
queries shape: (100, 64)


## 2. Build the Vamana index on the GPU

Key build parameters:

- `graph_degree` (R): maximum out-degree of each node in the graph.
- `visited_size` (L): size of the candidate list during graph construction.
- `alpha`: pruning aggressiveness (>= 1.0; larger keeps more long-range edges).

These map directly to the parameters described in the original Vamana / DiskANN paper.

In [7]:
graph_degree = 32
visited_size = 64

build_params = vamana.IndexParams(
    metric="sqeuclidean",
    graph_degree=graph_degree,
    visited_size=visited_size,
    alpha=1.2,
)

index = vamana.build(build_params, dataset)
print(index)

CuvsException: std::bad_alloc: CUDA error (failed to allocate 6400000 bytes) at: /tmp/conda-bld-output/bld/rattler-build_libcuvs-headers/host_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_/include/rmm/mr/cuda_memory_resource.hpp:51: cudaErrorIllegalAddress an illegal memory access was encountered

## 3. Serialize to the DiskANN in-memory format

`vamana.save(path, index)` writes two files compatible with DiskANN's `StaticMemoryIndex`:

- `<path>` — the Vamana graph file
- `<path>.data` — the (fp32/int8/uint8) dataset, in the DiskANN binary vector format

DiskANN's `StaticMemoryIndex` expects an `index_directory` plus an `index_prefix` and looks for files named `{index_prefix}` and `{index_prefix}.data`. We pick the prefix `ann` (DiskANN's default).

In [ ]:
index_dir = tempfile.mkdtemp(prefix="cuvs_vamana_diskann_")
index_prefix = "ann"
index_path = os.path.join(index_dir, index_prefix)

vamana.save(index_path, index, include_dataset=True)

print("Files written:")
for f in sorted(os.listdir(index_dir)):
    full = os.path.join(index_dir, f)
    print(f"  {f}  ({os.path.getsize(full)} bytes)")

## 4. Load the serialized index with `diskannpy` and search

Because cuVS does not write DiskANN's optional `_metadata.bin`, we pass `distance_metric`, `vector_dtype` and `dimensions` explicitly. The metric must match the one used at build time — `sqeuclidean` in cuVS corresponds to `"l2"` in DiskANN.

In [ ]:
search_complexity = 64

diskann_index = diskannpy.StaticMemoryIndex(
    index_directory=index_dir,
    index_prefix=index_prefix,
    num_threads=0,
    initial_search_complexity=search_complexity,
    distance_metric="l2",
    vector_dtype=np.float32,
    dimensions=n_features,
)

neighbors, distances = diskann_index.batch_search(
    queries,
    k_neighbors=k,
    complexity=search_complexity,
    num_threads=0,
)

print("neighbors shape:", neighbors.shape)
print("distances shape:", distances.shape)
print("\nFirst query, top-{} neighbors:".format(k))
print("  ids      :", neighbors[0])
print("  distances:", distances[0])

## 5. Sanity check: recall against brute-force ground truth

We compute exact L2 nearest neighbors with NumPy and measure recall@k of the DiskANN search results.

In [ ]:
def brute_force_topk(data, queries, k):
    # squared L2 distances via (a - b)^2 = a^2 - 2 a.b + b^2
    d2 = (
        (queries ** 2).sum(axis=1, keepdims=True)
        - 2.0 * queries @ data.T
        + (data ** 2).sum(axis=1)[None, :]
    )
    return np.argpartition(d2, kth=k, axis=1)[:, :k]


gt = brute_force_topk(dataset, queries, k)

matches = 0
for i in range(n_queries):
    matches += len(set(gt[i].tolist()) & set(neighbors[i].tolist()))

recall = matches / (n_queries * k)
print(f"recall@{k} = {recall:.3f}")

## 6. Cleanup

In [ ]:
shutil.rmtree(index_dir, ignore_errors=True)
print(f"Removed {index_dir}")